# Phase 237 — Per-seed vs ensemble: does geometry + topology beat geometry for localization?

Phase 236's pre-specified primary comparison (seed-paired pooled DCC@4A, geometry+topology vs geometry) was null
(+0.0051, 3/8 seeds, p = 0.62 / 0.92), but the 8-seed *ensemble* comparison was +0.0254 (structure-bootstrap CI
+0.004 to +0.047). One ensemble per arm cannot separate a real ensemble-level benefit from seed luck, and the structure
bootstrap ignores seed variance entirely. This notebook reloads the 32 saved checkpoints (no retraining), rescoring the
identical 1,149 benchmark structures, and asks:

1. How does the geometry+topology - geometry difference change with ensemble size k (all C(8,k) seed subsets, paired)?
2. What is the 8-seed-ensemble difference when BOTH seeds and structures are resampled (two-way bootstrap)?

## Cell 1 — Install

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

## Cell 2 — Setup (Phase 236 cell, unchanged)

In [ ]:
import pickle

import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

geo_tr = torch.load(locate("phase229_geometry_features.pt"), map_location="cpu", weights_only=False)
GEOM_DIM = len(geo_tr["names"]); assert GEOM_DIM == 17
SIDE_DIM = TOPO_DIM + GEOM_DIM
g_mu = torch.from_numpy(np.asarray(geo_tr["mean"], np.float32)); g_sd = torch.from_numpy(np.asarray(geo_tr["std"], np.float32))
_all = dataset; dataset = []; _dropped = 0
for d in _all:
    k = (str(d.pdb_id), str(d.chain_id))
    if k not in geo_tr["feats"]:
        _dropped += 1; continue
    f = geo_tr["feats"][k]; assert f.shape == (d.x.size(0), GEOM_DIM)
    z = ((torch.from_numpy(f) - g_mu) / g_sd).clamp(-5, 5)      # label-free statistics
    d.x = torch.cat([d.x, z], dim=1); dataset.append(d)
print(f"training chains with geometry: {len(dataset):,} (dropped {_dropped}); node dim {dataset[0].x.size(1)} = {N_PRE} + {TOPO_DIM} + {GEOM_DIM}")

## Cell 3 — Model class (Phase 22/236, unchanged) and arm definitions

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

import time
ARMS = ["no-topo", "full", "geom", "geom+topo"]
TOPO_COLS = list(range(N_PRE, N_PRE + TOPO_DIM))
GEOM_COLS = list(range(N_PRE + TOPO_DIM, N_PRE + SIDE_DIM))
KEEP = {"no-topo": [], "full": TOPO_COLS, "geom": GEOM_COLS, "geom+topo": TOPO_COLS + GEOM_COLS}

## Cell 4 — Load the 32 Phase 236 checkpoints

In [ ]:
models = {a: {} for a in ARMS}
for arm in ARMS:
    for seed in range(8):
        fn = f"loc236_{arm.replace('+','_')}_seed{seed}.pt"
        state = torch.load(locate(fn), map_location=DEVICE)
        m = LocalizingEGNN(N_PRE, SIDE_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
        models[arm][seed] = m
print({a: len(v) for a, v in models.items()}, "checkpoints loaded")

## Cell 5 — Benchmark features + geometry, and scoring (Phase 236 cells, unchanged)

In [ ]:
import pickle
with open(locate("phase15_raw_features.pkl"), "rb") as fh:
    raw_cache = pickle.load(fh)
geo_bm = torch.load(locate("phase235_benchmark_geometry.pt"), map_location="cpu", weights_only=False)
blob6b_bench = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
mu_b, sd_b = blob6b_bench["topo_mean"], blob6b_bench["topo_std"]
del blob6b_bench
valid_keys = {k for k in raw_cache if k in geo_bm["feats"]}
print(f"benchmark chains: {len(raw_cache):,} cached, {len(valid_keys):,} with geometry (dropped {len(raw_cache)-len(valid_keys)} for every arm)")

In [ ]:
from collections import defaultdict

def bench_x(entry, key, arm):
    x = torch.from_numpy(entry["x"]).clone()
    topo = (x[:, N_PRE:] - torch.from_numpy(mu_b)) / torch.from_numpy(sd_b)
    g = ((torch.from_numpy(geo_bm["feats"][key]) - g_mu) / g_sd).clamp(-5, 5)
    x = torch.cat([x[:, :N_PRE], topo, g], dim=1)
    mask = torch.ones(x.size(1), dtype=torch.bool); mask[N_PRE:] = False; mask[KEEP[arm]] = True
    x[:, ~mask] = 0.0
    return x

@torch.no_grad()
def predict_all_seeds(entry, key, arm):
    d = type("D", (), {})()
    d.x = bench_x(entry, key, arm).to(DEVICE); d.pos = torch.from_numpy(entry["pos"]).to(DEVICE)
    d.edge_index = torch.from_numpy(entry["edge_index"]).to(DEVICE); d.edge_attr = torch.from_numpy(entry["edge_attr"]).to(DEVICE)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long, device=DEVICE)
    offset = entry["ca"].mean(axis=0); out = {}
    for seed, m in models[arm].items():
        lg, pc = m(d)
        out[seed] = {"prob_sum": float(torch.sigmoid(lg).sum().cpu()), "center": pc.cpu().numpy()[0] + offset}
    return out

def dcc_dist(pred_center, ligs):
    if pred_center is None or not ligs: return None
    best = np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        best = min(best, float(np.linalg.norm(pred_center - lig_atoms.mean(axis=0))))
    return best if np.isfinite(best) else None

t0 = time.time()
by_structure = defaultdict(list)
for key in valid_keys: by_structure[key[:2]].append((key[2], key))
out = {a: {} for a in ARMS}; ligs_of = {}
for skey, chain_list in by_structure.items():
    ligs_of[skey] = raw_cache[chain_list[0][1]]["ligs"]
    for arm in ARMS:
        out[arm][skey] = [(cid, predict_all_seeds(raw_cache[k], k, arm)) for cid, k in chain_list]
print(f"inference (8 seeds x {len(ARMS)} arms x {len(by_structure)} structures) in {(time.time()-t0)/60:.1f} min")

def hits_for(arm, seeds, src=None):
    """dict structure -> 0/1 hit, ensemble over `seeds`, best chain by mean summed probability."""
    res = {}
    for skey, chain_list in out[arm].items():
        if src is not None and skey[0] != src: continue
        best_conf, best_center = -np.inf, None
        for cid, per_seed in chain_list:
            cs = [per_seed[s]["center"] for s in seeds if s in per_seed]; fs = [per_seed[s]["prob_sum"] for s in seeds if s in per_seed]
            if not cs: continue
            c, f = np.mean(cs, axis=0), np.mean(fs)
            if f > best_conf: best_conf, best_center = f, c
        dd = dcc_dist(best_center, ligs_of[skey])
        if dd is None: continue
        res[skey] = int(dd < 4.0)
    return res

def rate(h): return float(np.mean(list(h.values()))) if h else float("nan")

## Cell 6 — Sanity: reproduce Phase 236's per-seed and ensemble numbers

In [ ]:
per_seed = {a: np.array([rate(hits_for(a, (s,))) for s in range(8)]) for a in ARMS}
ens = {a: rate(hits_for(a, tuple(range(8)))) for a in ARMS}
print({a: round(float(per_seed[a].mean()), 4) for a in ARMS})
print({a: round(ens[a], 4) for a in ARMS})
# Phase 236 reported per-seed means no-topo .4441 full .4990 geom .5296 geom+topo .5347; ensembles .507 .5333 .5552 .5806
assert abs(per_seed["geom"].mean() - 0.5296) < 0.002 and abs(ens["geom+topo"] - 0.5806) < 0.002, "does not reproduce Phase 236"
print("reproduces Phase 236")

## Cell 7 — Ensemble-size analysis: all seed subsets of each size, paired across arms

In [ ]:
from itertools import combinations
import pandas as pd
COMPS = [("geom+topo", "geom"), ("full", "no-topo"), ("geom", "full"), ("geom", "no-topo")]
rows = []
for a, b in COMPS:
    for k in (1, 2, 3, 4, 6, 8):
        ds = np.array([rate(hits_for(a, S)) - rate(hits_for(b, S)) for S in combinations(range(8), k)])
        rows.append(dict(a=a, b=b, k=k, n_subsets=len(ds), mean_delta=float(ds.mean()), sd=float(ds.std()), frac_positive=float((ds > 0).mean())))
df_k = pd.DataFrame(rows); df_k.to_csv("/kaggle/working/phase237_ensemble_size.csv", index=False)
print(df_k.round(4).to_string(index=False))
# two disjoint 4-seed ensembles per arm (35 ways to split 8 seeds into two halves)
print("\nDisjoint 4+4 splits, geom+topo - geom (each half is an independent 4-model ensemble):")
halves = []
for S in combinations(range(8), 4):
    if 0 not in S: continue
    T = tuple(x for x in range(8) if x not in S)
    for H in (S, T): halves.append(rate(hits_for("geom+topo", H)) - rate(hits_for("geom", H)))
halves = np.array(halves)
print(f"  {len(halves)} half-ensemble deltas: mean {halves.mean():+.4f}  sd {halves.std():.4f}  positive {int((halves>0).sum())}/{len(halves)}")

## Cell 8 — Two-way bootstrap (seeds x structures) of the 8-seed-ensemble difference

In [ ]:
rng = np.random.default_rng(237)
B = 1000
res = {}
for a, b in COMPS:
    deltas = []
    for _ in range(B):
        seeds = tuple(int(x) for x in rng.integers(0, 8, size=8))          # resample the 8 seeds with replacement
        ha, hb = hits_for(a, seeds), hits_for(b, seeds)
        keys = sorted(set(ha) & set(hb)); va = np.array([ha[k] for k in keys]); vb = np.array([hb[k] for k in keys])
        idx = rng.integers(0, len(keys), size=len(keys))                     # resample structures
        deltas.append(va[idx].mean() - vb[idx].mean())
    deltas = np.array(deltas)
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    res[f"{a} - {b}"] = dict(mean=float(deltas.mean()), lo=float(lo), hi=float(hi), frac_pos=float((deltas > 0).mean()))
    print(f"{a+' - '+b:<22} two-way bootstrap ensemble delta {deltas.mean():+.4f}  95% [{lo:+.4f}, {hi:+.4f}]  P(delta>0) = {(deltas>0).mean():.3f}")
json.dump(res, open("/kaggle/working/phase237_twoway_bootstrap.json", "w"), indent=2)